# 03 — RAG as an MCP capability

**Learning goal:** put profile-aware retrieval behind genuine MCP tools, demonstrate protocol discovery without a model, then compare deterministic retrieve-then-answer with model-selected retrieval.

**Prerequisites:** Python 3 and this notebook inside `demos/`. Run cells top to bottom and choose `PROFILE="onia"` or `"devtalks"`. Installation, configuration, server creation, `list_tools`, and `list_sources` need neither LM Studio nor internet. Index construction is lazy: only `search_knowledge` contacts LM Studio for embeddings. The deterministic and agentic paths are separately opt-in, and both are disabled by default. Re-running factory cells creates fresh servers, avoiding duplicate registrations.

In [ ]:
%pip install -q openai==2.53.0 chromadb==1.5.9 "mcp[cli]==2.0.0"

## 1. Configuration and profile
Editable defaults can be overridden by environment variables. Document discovery begins at `Path.cwd()` and fails clearly when Jupyter starts outside the suite.

In [ ]:
import os
from pathlib import Path

OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "http://127.0.0.1:1234/v1")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "lm-studio")
CHAT_MODEL = os.getenv("CHAT_MODEL", "qwen/qwen3.5-9b")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-qwen3-embedding-4b")
PROFILE = os.getenv("PROFILE", "devtalks").lower()
TOP_K = int(os.getenv("TOP_K", "3"))
RUN_DETERMINISTIC_DEMO = False  # embeddings + chat through LM Studio
RUN_AGENTIC_DEMO = False       # tool calling + embeddings + chat through LM Studio

if PROFILE not in {"onia", "devtalks"}:
    raise ValueError("PROFILE must be 'onia' or 'devtalks'.")
if TOP_K <= 0:
    raise ValueError("TOP_K must be a positive integer.")

def find_demo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, cwd / "demos", *cwd.parents]:
        if (candidate / "documents" / "shared" / "mcp_plus_rag.md").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find demos/documents. Launch Jupyter from the presentations root or demos directory."
    )

DEMO_ROOT = find_demo_root()
DOCUMENTS_ROOT = DEMO_ROOT / "documents"
print(f"Profile: {PROFILE} | documents: {DOCUMENTS_ROOT}")

## 2. Build the selected corpus manifest — **offline**
The ONIA profile uses its MCP-RAG-specific model note. The DevTalks profile uses the broader shared model note. File contents are loaded only when the lazy index is first searched.

In [ ]:
PROFILE_DOCUMENTS = {
    "onia": [
        "shared/lm_studio.md",
        "shared/mcp_overview.md",
        "shared/mcp_plus_rag.md",
        "onia/qwen_models_mcp_rag.md",
        "onia/onia_conference.md",
    ],
    "devtalks": [
        "shared/lm_studio.md",
        "shared/mcp_overview.md",
        "shared/mcp_plus_rag.md",
        "shared/qwen_models.md",
        "devtalks/devtalks_conference.md",
    ],
}
selected_paths = [DOCUMENTS_ROOT / relative for relative in PROFILE_DOCUMENTS[PROFILE]]
missing = [path for path in selected_paths if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Missing demo documents: {missing}")
for path in selected_paths:
    print(path.relative_to(DOCUMENTS_ROOT).as_posix())

## 3. Embedding adapter and lazy MCP server
`list_sources` reads only the manifest. `search_knowledge` triggers the one-time in-memory Chroma build and therefore requires LM Studio's embedding model. A fresh closure holds independent state on every factory call.

In [ ]:
from typing import Any

import chromadb
from chromadb.api.types import Documents, EmbeddingFunction, Embeddings
from mcp.server import MCPServer
from openai import OpenAI

class OpenAIEmbeddingFunction(EmbeddingFunction[Documents]):
    def __init__(self, client: OpenAI, model: str) -> None:
        self.client = client
        self.model = model

    def __call__(self, input: Documents) -> Embeddings:
        response = self.client.embeddings.create(model=self.model, input=list(input))
        return [item.embedding for item in response.data]

def make_rag_server() -> MCPServer:
    server = MCPServer(f"notebook-mcp-rag-{PROFILE}")
    state: dict[str, Any] = {"collection": None, "chroma_client": None}

    def ensure_index():
        if state["collection"] is not None:
            return state["collection"]
        llm = OpenAI(base_url=OPENAI_BASE_URL, api_key=OPENAI_API_KEY)
        embedder = OpenAIEmbeddingFunction(llm, EMBEDDING_MODEL)
        chroma_client = chromadb.EphemeralClient()
        collection = chroma_client.get_or_create_collection(
            name=f"notebook_mcp_rag_{PROFILE}", embedding_function=embedder
        )
        sources = [path.relative_to(DOCUMENTS_ROOT).as_posix() for path in selected_paths]
        collection.add(
            ids=[f"doc-{index}" for index in range(len(selected_paths))],
            documents=[path.read_text(encoding="utf-8") for path in selected_paths],
            metadatas=[{"source": source} for source in sources],
        )
        state.update(collection=collection, chroma_client=chroma_client)
        print(f"Lazy index built with {collection.count()} documents.")
        return collection

    @server.tool()
    def search_knowledge(query: str, k: int = TOP_K) -> str:
        """Search the selected local corpus and return source-labelled passages."""
        if k <= 0:
            return "k must be a positive integer."
        collection = ensure_index()
        result = collection.query(query_texts=[query], n_results=min(k, collection.count()))
        return "\n\n---\n\n".join(
            f"[source: {meta['source']}] (distance={distance:.4f})\n{text.strip()}"
            for text, meta, distance in zip(
                result["documents"][0], result["metadatas"][0], result["distances"][0]
            )
        )

    @server.tool()
    def list_sources() -> list[str]:
        """List source filenames in the selected profile without building the index."""
        return [path.relative_to(DOCUMENTS_ROOT).as_posix() for path in selected_paths]

    return server

rag_server = make_rag_server()
print("Fresh lazy MCP-RAG server created; no model calls made.")

## 4. MCP discovery — **offline, no LM Studio**
This genuine MCP exchange lists schemas and calls only `list_sources`; lazy retrieval remains untouched. Jupyter's top-level `await` keeps the protocol flow visible.

In [ ]:
from mcp import ClientSession
from mcp.client._memory import InMemoryTransport

def tool_result_text(result: Any) -> str:
    return "\n".join(getattr(block, "text", str(block)) for block in result.content).strip()

async with InMemoryTransport(rag_server, raise_exceptions=True) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        tools = await session.list_tools()
        print("Advertised:", [tool.name for tool in tools.tools])
        sources_result = await session.call_tool("list_sources", {})
        print("Sources via MCP:", tool_result_text(sources_result))

## 5. Deterministic retrieve-then-answer — **opt-in LM Studio**
This path always calls `search_knowledge` once and then gives its passages to the chat model. It is useful when reliable grounding matters more than model autonomy.

In [ ]:
QUESTION = (
    "What is ONIA, and how does MCP combine with RAG in this demo?"
    if PROFILE == "onia"
    else "What is DevTalks, and how does MCP combine with RAG in this demo?"
)

if RUN_DETERMINISTIC_DEMO:
    deterministic_server = make_rag_server()
    async with InMemoryTransport(deterministic_server, raise_exceptions=True) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool("search_knowledge", {"query": QUESTION, "k": TOP_K})
            context = tool_result_text(result)
            print(context[:1200])
    llm = OpenAI(base_url=OPENAI_BASE_URL, api_key=OPENAI_API_KEY)
    response = llm.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": "Answer only from context; cite source filenames."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {QUESTION}"},
        ],
        temperature=0.2,
    )
    print("\nAnswer:\n", response.choices[0].message.content or "")
else:
    print("Skipped deterministic path. Set RUN_DETERMINISTIC_DEMO=True to enable it.")

## 6. Agentic retrieval — **opt-in LM Studio**
Here MCP advertises the same tools to the model. The model decides when to search, while the system prompt requires grounded answers and filename citations.

In [ ]:
import json

def openai_tool_schemas(mcp_tools: Any) -> list[dict]:
    return [{
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description or "",
            "parameters": tool.inputSchema or {"type": "object"},
        },
    } for tool in mcp_tools.tools]

if RUN_AGENTIC_DEMO:
    llm = OpenAI(base_url=OPENAI_BASE_URL, api_key=OPENAI_API_KEY)
    agent_server = make_rag_server()
    async with InMemoryTransport(agent_server, raise_exceptions=True) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            schemas = openai_tool_schemas(await session.list_tools())
            messages = [
                {"role": "system", "content": (
                    "Consult search_knowledge for local-demo questions. Answer only from retrieved "
                    "passages, cite source filenames, and say when the corpus lacks the answer."
                )},
                {"role": "user", "content": QUESTION},
            ]
            for step in range(5):
                response = llm.chat.completions.create(
                    model=CHAT_MODEL, messages=messages, tools=schemas,
                    tool_choice="auto", temperature=0.2,
                )
                message = response.choices[0].message
                calls = message.tool_calls or []
                if not calls:
                    print(message.content or "")
                    break
                messages.append({"role": "assistant", "content": message.content or "",
                                 "tool_calls": [call.model_dump() for call in calls]})
                for call in calls:
                    arguments = json.loads(call.function.arguments or "{}")
                    result = await session.call_tool(call.function.name, arguments)
                    text = tool_result_text(result)
                    print(f"tool: {call.function.name}({arguments}) -> {text[:180]}…")
                    messages.append({"role": "tool", "tool_call_id": call.id, "content": text})
            else:
                print("Stopped after five tool-selection steps.")
else:
    print("Skipped agentic path. Set RUN_AGENTIC_DEMO=True to enable it.")